# MASt3R Orthophoto — Palazzo Adriatica

Pipeline 3D-from-photos su Colab (GPU consigliata).

In [ ]:
!nvidia-smi || echo 'NO GPU'

In [ ]:
%%capture
!pip install --quiet einops trimesh safetensors huggingface_hub roma scikit-learn opencv-python pillow tqdm

In [ ]:
import os, sys
if not os.path.exists('/content/mast3r'):
    !git clone --recursive https://github.com/naver/mast3r /content/mast3r
sys.path.insert(0, '/content/mast3r')
sys.path.insert(0, '/content/mast3r/dust3r')
%cd /content/mast3r
!git submodule update --init --recursive --quiet

In [ ]:
from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(
    repo_id='naver/MASt3R_ViTLarge_BaseDecoder_512_catmlspretrained',
    filename='MASt3R_ViTLarge_BaseDecoder_512_catmlspretrained.pth',
    cache_dir='/content/checkpoints')
print('Checkpoint:', ckpt_path)

In [ ]:
import os, getpass, urllib.request
REPO = 'robertooleotto/Acrobatica'
BRANCH = 'samples/palazzo-adriatica'
FOLDER = 'samples/palazzo-adriatica'
IMAGES = ['IMG_2974.jpeg','IMG_2975.jpeg','IMG_2976.jpeg','IMG_2977.jpeg','IMG_2978.jpeg','IMG_2979.jpeg']
os.makedirs('/content/photos', exist_ok=True)
token = getpass.getpass('GitHub PAT (vuoto se repo pubblico): ')
headers = {'Authorization': f'Bearer {token}'} if token else {}
for img in IMAGES:
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{FOLDER}/{img}'
    req = urllib.request.Request(url, headers=headers)
    out = f'/content/photos/{img}'
    with urllib.request.urlopen(req) as r, open(out,'wb') as f:
        f.write(r.read())
    print(' ', img, os.path.getsize(out)//1024, 'KB')

In [ ]:
import torch
from mast3r.model import AsymmetricMASt3R
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
model = AsymmetricMASt3R.from_pretrained(ckpt_path).to(device).eval()

In [ ]:
import glob
from dust3r.inference import inference
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
image_paths = sorted(glob.glob('/content/photos/*.jpeg'))
images = load_images(image_paths, size=512)
pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
print('Coppie:', len(pairs))
output = inference(pairs, model, device, batch_size=1, verbose=True)

In [ ]:
from mast3r.cloud_opt.sparse_ga import sparse_global_alignment
os.makedirs('/content/cache', exist_ok=True)
scene = sparse_global_alignment(image_paths, pairs, '/content/cache', model,
    lr1=0.07, niter1=500, lr2=0.014, niter2=200, device=device, matching_conf_thr=5.0)

In [ ]:
import numpy as np, trimesh
pts3d, depthmaps, confs = scene.get_dense_pts3d(clean_depth=True)
imgs = scene.imgs
msk = scene.get_masks()
all_pts, all_col = [], []
for i in range(len(imgs)):
    m = msk[i].cpu().numpy() if hasattr(msk[i],'cpu') else msk[i]
    p = pts3d[i].detach().cpu().numpy() if hasattr(pts3d[i],'detach') else np.asarray(pts3d[i])
    c = imgs[i].cpu().numpy() if hasattr(imgs[i],'cpu') else np.asarray(imgs[i])
    all_pts.append(p.reshape(-1,3)[m.reshape(-1)])
    all_col.append(c.reshape(-1,3)[m.reshape(-1)])
P = np.concatenate(all_pts, axis=0); C = np.concatenate(all_col, axis=0)
C8 = (np.clip(C,0,1)*255).astype(np.uint8) if C.max()<=1.001 else (((np.clip(C,-1,1)+1)/2)*255).astype(np.uint8)
trimesh.PointCloud(P, colors=C8).export('/content/palazzo_cloud.ply')
print('Punti:', P.shape[0])

In [ ]:
centroid = P.mean(axis=0)
_,_,Vt = np.linalg.svd(P - centroid, full_matrices=False)
normal = Vt[-1]
if np.dot(normal, [0,0,1]) < 0: normal = -normal
right = np.cross([0,-1,0], normal); right /= np.linalg.norm(right)
up = np.cross(normal, right); up /= np.linalg.norm(up)
rel = P - centroid
u = rel @ right; v = rel @ up; depth = rel @ normal
u_lo,u_hi = np.percentile(u,[1,99]); v_lo,v_hi = np.percentile(v,[1,99])
OUT_W = 2400; scale = OUT_W/(u_hi-u_lo); OUT_H = int((v_hi-v_lo)*scale)
ortho = np.zeros((OUT_H,OUT_W,3), dtype=np.uint8)
best = np.full((OUT_H,OUT_W), -np.inf, dtype=np.float32)
px = ((u-u_lo)*scale).astype(np.int32); py = OUT_H-1-((v-v_lo)*scale).astype(np.int32)
valid = (px>=0)&(px<OUT_W)&(py>=0)&(py<OUT_H)
px,py,d,col = px[valid],py[valid],depth[valid],C8[valid]
for i in range(len(px)):
    if d[i] > best[py[i],px[i]]: best[py[i],px[i]] = d[i]; ortho[py[i],px[i]] = col[i]
import cv2
cv2.imwrite('/content/orthophoto.jpg', cv2.cvtColor(ortho, cv2.COLOR_RGB2BGR))
mask = (best == -np.inf).astype(np.uint8)*255
filled = cv2.inpaint(ortho, mask, 3, cv2.INPAINT_TELEA)
cv2.imwrite('/content/orthophoto_filled.jpg', cv2.cvtColor(filled, cv2.COLOR_RGB2BGR))
from IPython.display import Image, display
display(Image('/content/orthophoto_filled.jpg', width=900))

In [ ]:
from google.colab import files
files.download('/content/orthophoto.jpg')
files.download('/content/orthophoto_filled.jpg')
files.download('/content/palazzo_cloud.ply')